In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only — redo Runtime > Change runtime type")

In [ ]:
!pip install ultralytics -q

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="urbcWxG2W5vUlyLehwWp")
project = rf.workspace("roboflow-jvuqo").project("football-players-detection-3zvbc")
version = project.version(15)
dataset = version.download("yolov8")


In [ ]:
data_yaml = "/content/football-players-detection-15/data.yaml"
print(open(data_yaml).read())

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
results = model.train(
    data="/content/football-players-detection-15/data.yaml",
    epochs=30,
    imgsz=640,
    patience=10,
)

In [ ]:
from google.colab import files
files.download("/content/runs/detect/train/weights/best.pt")

In [ ]:
import glob, os
import matplotlib.pyplot as plt
import cv2

In [ ]:
model = YOLO("/content/runs/detect/train/weights/best.pt")
names = model.names
CLASS_COLORS = {
    "player":     (255, 180,  0),   # blue-ish
    "goalkeeper": (0,   255,  0),   # green
    "referee":    (0,   255, 255),  # yellow
    "ball":       (0,   100, 255),  # orange
}

def annotate(img, res):
    for box in res.boxes:
        cls_name = names[int(box.cls[0])]
        score = float(box.conf[0])
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        color = CLASS_COLORS.get(cls_name, (200, 200, 200))

        if cls_name == "ball":
            cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
            cv2.circle(img, (cx, cy), max(6, (x2 - x1) // 2), color, 2)
            continue

        cx = (x1 + x2) // 2
        feet_y = y2
        w = x2 - x1
        axes = (max(int(w * 0.5), 8), max(int(w * 0.22), 4))
        cv2.ellipse(img, (cx, feet_y), axes,
                    angle=0, startAngle=-45, endAngle=235,
                    color=color, thickness=2, lineType=cv2.LINE_AA)

        label = f"{cls_name} {score:.2f}"
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        ly = feet_y + th + 6
        cv2.rectangle(img, (cx - tw // 2 - 2, ly - th - 2),
                      (cx + tw // 2 + 2, ly + 2), color, -1)
        cv2.putText(img, label, (cx - tw // 2, ly),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
    return img

test_dir = "/content/football-players-detection-15/test/images"
test_imgs = sorted(glob.glob(os.path.join(test_dir, "*.jpg")))

for path in test_imgs[:6]:
    res = model.predict(path, conf=0.25, imgsz=640, verbose=False)[0]
    img = annotate(cv2.imread(path), res)
    plt.figure(figsize=(14, 8))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.axis("off"); plt.title(os.path.basename(path)); plt.show()

In [ ]:
from sklearn.cluster import KMeans
import numpy as np

In [ ]:
!pip install transformers umap-learn -q

In [ ]:
import torch
import numpy as np
from transformers import AutoProcessor, AutoModel
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"

# SigLIP vision model
SIGLIP_ID = "google/siglip-base-patch16-224"
siglip_processor = AutoProcessor.from_pretrained(SIGLIP_ID)
siglip_model = AutoModel.from_pretrained(SIGLIP_ID).to(device).eval()

@torch.no_grad()
def embed_crops(crops):
    pil = [Image.fromarray(c[:, :, ::-1]) for c in crops]      # BGR->RGB
    inputs = siglip_processor(images=pil, return_tensors="pt").to(device)

    out = siglip_model.get_image_features(**inputs)
    # Some transformers versions return a tensor, others an output object.
    feats = out if isinstance(out, torch.Tensor) else out.pooler_output

    feats = feats / feats.norm(dim=-1, keepdim=True)
    return feats.cpu().numpy()

In [ ]:
import glob, os, cv2
from ultralytics import YOLO

model = YOLO("/content/runs/detect/train/weights/best.pt")
names = model.names

test_dir = "/content/football-players-detection-15/test/images"
test_imgs = sorted(glob.glob(os.path.join(test_dir, "*.jpg")))

all_crops = []          # every player crop, across all frames
per_image = {}          # path -> list of (cls, cx, feet_y, w, crop_index_or_None)

for path in test_imgs:
    img = cv2.imread(path)
    res = model.predict(path, conf=0.25, imgsz=640, verbose=False)[0]
    items = []
    for box in res.boxes:
        cls = names[int(box.cls[0])]
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        cx, feet_y, w = (x1 + x2)//2, y2, (x2 - x1)
        idx = None
        if cls == "player":
            crop = img[max(y1,0):y2, max(x1,0):x2]
            if crop.size > 0:
                idx = len(all_crops)
                all_crops.append(crop)
        items.append((cls, cx, feet_y, w, idx))
    per_image[path] = items

print(f"{len(all_crops)} player crops collected")

In [ ]:
import umap
from sklearn.cluster import KMeans

# SigLIP embeddings for every player crop (batch through to save memory)
emb = []
B = 64
for i in range(0, len(all_crops), B):
    emb.append(embed_crops(all_crops[i:i+B]))
emb = np.concatenate(emb, axis=0)
print("embeddings:", emb.shape)

# UMAP -> 3D
reducer = umap.UMAP(n_components=3, n_neighbors=15, min_dist=0.0, random_state=0)
reduced = reducer.fit_transform(emb)

# KMeans -> 2 teams, fit globally
km = KMeans(n_clusters=2, n_init=10, random_state=0).fit(reduced)
team_of_crop = km.labels_          # team label per crop index
print("cluster sizes:", np.bincount(team_of_crop))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

TEAM_COLORS = [(255,0,0),(0,0,255)]
GK_COLOR, REF_COLOR, BALL_COLOR = (0,255,0),(0,255,255),(0,165,255)

def draw_ellipse(img, cx, feet_y, w, color, label):
    axes=(max(int(w*0.5),8),max(int(w*0.22),4))
    cv2.ellipse(img,(cx,feet_y),axes,0,-45,235,color,2,cv2.LINE_AA)
    (tw,th),_=cv2.getTextSize(label,cv2.FONT_HERSHEY_SIMPLEX,0.45,1)
    ly=feet_y+th+5
    cv2.rectangle(img,(cx-tw//2-2,ly-th-2),(cx+tw//2+2,ly+2),color,-1)
    cv2.putText(img,label,(cx-tw//2,ly),cv2.FONT_HERSHEY_SIMPLEX,0.45,(0,0,0),1,cv2.LINE_AA)

for path in test_imgs[:6]:
    img = cv2.imread(path)
    # gather THIS frame's player crop indices + embeddings
    idxs = [idx for (cls,_,_,_,idx) in per_image[path] if cls=="player" and idx is not None]
    if len(idxs) >= 2:
        local_emb = reduced[idxs]                       # use the UMAP features
        local_km = KMeans(n_clusters=2, n_init=10, random_state=0).fit(local_emb)
        team_map = {idx: int(t) for idx, t in zip(idxs, local_km.labels_)}
    else:
        team_map = {i: 0 for i in idxs}

    for cls, cx, feet_y, w, idx in per_image[path]:
        if cls=="player" and idx is not None:
            draw_ellipse(img, cx, feet_y, w, TEAM_COLORS[team_map[idx]], f"team {team_map[idx]}")
        elif cls=="goalkeeper":
            draw_ellipse(img, cx, feet_y, w, GK_COLOR, "GK")
        elif cls=="referee":
            draw_ellipse(img, cx, feet_y, w, REF_COLOR, "ref")
        elif cls=="ball":
            cv2.circle(img,(cx,feet_y),max(6,w//2),BALL_COLOR,2)
    plt.figure(figsize=(14,8))
    plt.imshow(cv2.cvtColor(img,cv2.COLOR_BGR2RGB))
    plt.axis("off"); plt.title(os.path.basename(path)); plt.show()

In [ ]:
from google.colab import files
files.download("/content/runs/detect/train/weights/best.pt")

In [ ]:
import os
out_dir = "/content/results"
os.makedirs(out_dir, exist_ok=True)

for path in test_imgs:
    img = cv2.imread(path)
    idxs = [idx for (cls,_,_,_,idx) in per_image[path] if cls=="player" and idx is not None]
    if len(idxs) >= 2:
        from sklearn.cluster import KMeans
        lk = KMeans(2, n_init=10, random_state=0).fit(reduced[idxs])
        tmap = {i:int(t) for i,t in zip(idxs, lk.labels_)}
    else:
        tmap = {i:0 for i in idxs}
    for cls, cx, feet_y, w, idx in per_image[path]:
        if cls=="player" and idx is not None:
            draw_ellipse(img, cx, feet_y, w, TEAM_COLORS[tmap[idx]], f"team {tmap[idx]}")
        elif cls=="goalkeeper": draw_ellipse(img, cx, feet_y, w, GK_COLOR, "GK")
        elif cls=="referee":    draw_ellipse(img, cx, feet_y, w, REF_COLOR, "ref")
        elif cls=="ball":       cv2.circle(img,(cx,feet_y),max(6,w//2),BALL_COLOR,2)
    cv2.imwrite(os.path.join(out_dir, os.path.basename(path)), img)

!zip -qr /content/results.zip /content/results
files.download("/content/results.zip")